In [1]:
!nvidia-smi

Wed Jul  8 06:20:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers datasets accelerate evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

# use small subset to save time/compute
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [12]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="distilbert-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    fp16=True,                  # faster on T4, less memory
    logging_steps=50,
    push_to_hub=False,
)

In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.438595,0.310172,0.870000
2,0.185534,0.360937,0.864000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.330332706451416, metrics={'train_runtime': 45.895, 'train_samples_per_second': 87.155, 'train_steps_per_second': 5.447, 'total_flos': 264934797312000.0, 'train_loss': 0.330332706451416, 'epoch': 2.0})

In [16]:
trainer.save_model("distilbert-stanfordnlp/imdb-final")
tokenizer.save_pretrained("distilbert-stanfordnlp/imdb-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert-stanfordnlp/imdb-final/tokenizer_config.json',
 'distilbert-stanfordnlp/imdb-final/tokenizer.json')

In [17]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/abdulhadi19306/distilbert-imdb/commit/ea36b1799fd457d0ab7c02dc7fc2415ba0fc8efd', commit_message='End of training', commit_description='', oid='ea36b1799fd457d0ab7c02dc7fc2415ba0fc8efd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/abdulhadi19306/distilbert-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='abdulhadi19306/distilbert-imdb'), pr_revision=None, pr_num=None)

In [18]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.185534,0.360937,2,0.864000


{'eval_loss': 0.3609369993209839, 'eval_accuracy': 0.864}


In [19]:
full_test = dataset["test"].map(tokenize_fn, batched=True)
trainer.evaluate(eval_dataset=full_test)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.185534,0.341551,2,0.877080


{'eval_loss': 0.34155067801475525, 'eval_accuracy': 0.87708}

In [26]:
from transformers import pipeline

classifier = pipeline("text-classification", model="distilbert-stanfordnlp/imdb-final", tokenizer="distilbert-stanfordnlp/imdb-final")

texts = [
    "I still dont understand what happened in the movie."
]

results = classifier(texts)
print(results)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9042273759841919}]


In [27]:
import torch

text = "I still dont understand what happened in the movie."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)

# Move input tensors to the same device as the model
inputs = {key: val.to(model.device) for key, val in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=-1).item()
print("Positive" if pred == 1 else "Negative")

Negative


In [28]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=["Negative", "Positive"]))

              precision    recall  f1-score   support

    Negative       0.86      0.87      0.87       254
    Positive       0.86      0.86      0.86       246

    accuracy                           0.86       500
   macro avg       0.86      0.86      0.86       500
weighted avg       0.86      0.86      0.86       500

